# Thrust switchable-backend POC (one source → CPU or GPU)

Proves a **device-resident buffer** (`thrust::device_vector`) with chained ops + reduction
compiles from **one source** to either **GPU (nvcc/CUDA)** or **CPU (g++/host)** — no CUDA lock-in.

**Runtime → Change runtime type → GPU (T4)**, then Run all.
Both builds must print the SAME sum: **147432.281250** (matches the local MSVC CPU run).


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


### Clone the `thrust-backend` branch


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### GPU build (nvcc, CUDA device system) + run


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda pure/thrust_poc.cpp -o poc_gpu
!./poc_gpu


### CPU build — SAME source, host device system (g++) + run


In [ ]:
!g++ -O2 -std=c++17 -I/usr/local/cuda/include -I/usr/local/cuda/include/cccl \
     -DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_CPP pure/thrust_poc.cpp -o poc_cpu
!./poc_cpu


---
**Expected:** both cells print `sum(0.5*SiLU(ramp)) = 147432.281250` — one prints
`backend: GPU (CUDA)`, the other `backend: CPU (host device system)`. Same source, same
result, switchable at compile time. (Multicore CPU: swap `_CPP` → `_OMP` and add `-fopenmp`.)
